# 在Pandas中操作数据：



NumPy 的一个优势在于，它允许我们快速执行逐元素操作，包括基本的算术运算（加法、减法、乘法等）以及更复杂的运算（三角函数、指数和对数函数等）。

Pandas 从 NumPy 继承了许多功能，其中在 [Computation on NumPy Arrays: Universal Functions](02.03-Computation-on-arrays-ufuncs.ipynb) 中介绍的 ufunc 是关键。

然而，Pandas 还包含了一些有用的变化：对于像取反和三角函数这样的单目运算，这些 ufunc 将 *保留输出中的索引和列标签*；而对于如加法和乘法等双目运算，Pandas 会自动 *对齐索引* 在将对象传递给 ufunc 时。

这意味着，在 Pandas 中保持数据上下文并结合来自不同来源的数据——这两个任务在使用原始 NumPy 数组时可能容易出错——变得几乎是万无一失的。

此外，我们还会看到一维 `Series` 结构与二维 `DataFrame` 结构之间存在明确定义的操作。

## Ufuncs: 索引保留

由于 Pandas 旨在与 NumPy 一起使用，因此任何 NumPy ufunc 都可以作用于 Pandas 的 `Series` 和 `DataFrame` 对象。

让我们首先定义一个简单的 `Series` 和 `DataFrame` 来演示这一点。

In [1]:
import pandas as pd
import numpy as np

In [2]:
rng = np.random.default_rng(42)
ser = pd.Series(rng.integers(0, 10, 4))
ser

0    0
1    7
2    6
3    4
dtype: int64

In [3]:
df = pd.DataFrame(rng.integers(0, 10, (3, 4)),
                  columns=['A', 'B', 'C', 'D'])
df

,A,B,C,D
0,4,8,0,6
1,2,0,5,9
2,7,7,7,7


如果我们对这两个对象中的任一个应用NumPy的ufunc，结果将是另一个Pandas对象*并保留索引：*

In [4]:
np.exp(ser)

0       1.000000
1    1096.633158
2     403.428793
3      54.598150
dtype: float64

这对于更复杂的操作序列也是正确的：

In [5]:
np.sin(df * np.pi / 4)

,A,B,C,D
0,1.224647e-16,-2.449294e-16,0.000000,-1.000000
1,1.000000e+00,0.000000e+00,-0.707107,0.707107
2,-7.071068e-01,-7.071068e-01,-0.707107,-0.707107


任何在[NumPy数组计算：通用函数](02.03-Computation-on-arrays-ufuncs.ipynb)中讨论的ufuncs都可以以类似的方式使用。

## Ufuncs: 索引对齐

在对两个 `Series` 或 `DataFrame` 对象进行二元操作时，Pandas 会在执行操作的过程中自动对齐索引。

这在处理不完整数据时非常方便，正如我们接下来的一些例子所示。

### Series中的索引对齐

作为一个例子，假设我们正在合并两个不同的数据源，并希望仅找到按*面积*排名的前三个美国州和按*人口*排名的前三个美国州。

In [6]:
area = pd.Series({'Alaska': 1723337, 'Texas': 695662,
                  'California': 423967}, name='area')
population = pd.Series({'California': 39538223, 'Texas': 29145505,
                        'Florida': 21538187}, name='population')

让我们看看当我们将这些进行除法运算以计算人口密度时会发生什么：

In [7]:
population / area

Alaska              NaN
California    93.257784
Florida             NaN
Texas         41.896072
dtype: float64

结果数组包含两个输入数组索引的*并集*，这些索引可以直接从中确定。

In [8]:
area.index.union(population.index)

Index(['Alaska', 'California', 'Florida', 'Texas'], dtype='str')

任何一项如果没有一个或另一个的条目，则标记为 `NaN`，即“非数字”，这是 Pandas 标记缺失数据的方式（有关缺失数据的进一步讨论，请参见 [处理缺失数据](03.04-Missing-Values.ipynb)）。

这种索引匹配方法适用于 Python 的所有内置算术表达式；任何缺失值都被标记为 `NaN`。

In [9]:
A = pd.Series([2, 4, 6], index=[0, 1, 2])
B = pd.Series([1, 3, 5], index=[1, 2, 3])
A + B

0    NaN
1    5.0
2    9.0
3    NaN
dtype: float64

如果不希望使用 `NaN` 值，可以通过适当的对象方法来修改填充值，而不是使用运算符。

例如，调用 ``A.add(B)`` 相当于调用 ``A + B``，但允许为可能缺失的 ``A`` 或 ``B`` 中的任何元素显式指定填充值。

In [10]:
A.add(B, fill_value=0)

0    2.0
1    5.0
2    9.0
3    5.0
dtype: float64

### DataFrame中的索引对齐

在对`DataFrame`对象执行操作时，*列*和*索引*都会发生类似的对齐。

In [11]:
A = pd.DataFrame(rng.integers(0, 20, (2, 2)),
                 columns=['a', 'b'])
A

,a,b
0,10,2
1,16,9


In [12]:
B = pd.DataFrame(rng.integers(0, 10, (3, 3)),
                 columns=['b', 'a', 'c'])
B

,b,a,c
0,5,3,1
1,9,7,6
2,4,8,5


In [13]:
A + B

,a,b,c
0,13.0,7.0,NaN
1,23.0,18.0,NaN
2,NaN,NaN,NaN


请注意，无论两个对象中的索引顺序如何，索引都已正确对齐，并且结果中的索引是排序的。

与 `Series` 的情况一样，我们可以使用相关对象的算术方法，并传递任何所需的 `fill_value` 来替代缺失条目。

在这里，我们将用 `A` 中所有值的均值进行填充：

In [14]:
A.add(B, fill_value=A.values.mean())

,a,b,c
0,13.00,7.00,10.25
1,23.00,18.00,15.25
2,17.25,13.25,14.25


以下表格列出了Python运算符及其对应的Pandas对象方法：

| Python operator | Pandas method(s)                |
|-----------------|---------------------------------|
| `+`             | `add`                           |
| `-`             | `sub`, `subtract`               |
| `*`             | `mul`, `multiply`               |
| `/`             | `truediv`, `div`, `divide`      |
| `//`            | `floordiv`                      |
| `%`             | `mod`                           |
| `**`            | `pow`                           |


## Ufuncs: DataFrame与Series之间的操作

在进行`DataFrame`和`Series`之间的操作时，索引和列对齐同样得以保持，结果类似于二维数组与一维NumPy数组之间的操作。

考虑一个常见的操作，即计算一个二维数组与其某一行之间的差异。

In [15]:
A = rng.integers(10, size=(3, 4))
A

array([[4, 4, 2, 0],
       [5, 8, 0, 8],
       [8, 2, 6, 1]])

In [16]:
A - A[0]

array([[ 0,  0,  0,  0],
       [ 1,  4, -2,  8],
       [ 4, -2,  4,  1]])

根据NumPy的广播规则（见[数组计算：广播](02.05-Computation-on-arrays-broadcasting.ipynb)），二维数组与其某一行之间的减法操作是逐行进行的。

在Pandas中，默认情况下，该约定同样按行操作。

In [17]:
df = pd.DataFrame(A, columns=['Q', 'R', 'S', 'T'])
df - df.iloc[0]

,Q,R,S,T
0,0,0,0,0
1,1,4,-2,8
2,4,-2,4,1


如果您希望按列操作，可以使用前面提到的对象方法，同时指定 `axis` 关键字：

In [18]:
df.subtract(df['R'], axis=0)

,Q,R,S,T
0,0,0,-2,-4
1,-3,0,-8,0
2,6,0,4,-1


请注意，这些 `DataFrame`/`Series` 操作与之前讨论的操作类似，将自动对齐两个元素之间的索引。

In [19]:
halfrow = df.iloc[0, ::2]
halfrow

Q    4
S    2
Name: 0, dtype: int64

In [20]:
df - halfrow

,Q,R,S,T
0,0.0,NaN,0.0,NaN
1,1.0,NaN,-2.0,NaN
2,4.0,NaN,4.0,NaN


这种对索引和列的保存与对齐意味着，在Pandas中对数据进行操作时，将始终保持数据上下文，从而防止在处理原始NumPy数组中的异构和/或未对齐数据时可能出现的常见错误。